# CQT RCNN Trainer

L'exécution de ce notebook a pour prérequis :
- Le téléchargement du dataset `GuitarSet`,
- le démarrage de l'infrastructure docker,
- l'ingestion du dataset `GuitarSet`,
- le prétraitement du dataste `GuitarSet`.

Pour télécharger le dataset `GuitarSet`, utilisez la commande :
```bash
uv run ./audio_midi/main.py --download_datasets --no_idmt_smt_guitar
```

Pour démarrer l'infrastructure docker, utilisez la commande :
```bash
docker-compose up -d
```

Pour lancer la pipeline d'ingestion, utilisez la commande :
```bash
uv run ./audio_midi/main.py --ingest_guitar_set
```

Pour lancer la pipeline de prétraitement, utilisez la commande :
```bash
uv run ./audio_midi/main.py --preprocess_datasets --no_idmt_smt_guitar
```

## Imports

In [1]:
import sys
from pathlib import Path

APP_DIR = Path.cwd().parent
sys.path.append(APP_DIR.as_posix())

In [2]:
# Chemins
OUTPUT_DIR = APP_DIR / "output"
ARTIFACT_DIR = OUTPUT_DIR / "cqt_rcnn"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Imports graphiques
import matplotlib.pyplot as plt
import seaborn as sns

# Accessibilité : Daltonisme, Dyslexie, Confort Visuel
sns.set_theme(
    style="whitegrid",
    palette="colorblind",
    context="notebook",
)

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.family": "Arial",
        "font.size": 12,
        "axes.titlesize": 15,
        "axes.titleweight": "bold",
        "axes.labelsize": 13,
        "axes.labelweight": "medium",
        "axes.edgecolor": "black",
        "axes.linewidth": 1.2,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "lines.linewidth": 2.2,
        "lines.markersize": 7,
        "legend.fontsize": 11,
        "legend.frameon": True,
        "legend.framealpha": 0.95,
        "grid.linestyle": ":",
        "grid.linewidth": 0.8,
        "grid.alpha": 0.6,
    }
)

COLORBLIND_PALETTE = sns.color_palette("colorblind")

In [4]:
import os
import json
import warnings
import logging
from datetime import datetime
from time import perf_counter
import functools

import numpy as np
import pandas as pd

import tensorflow as tf
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    hamming_loss,
    accuracy_score,
)

import mlflow
from mlflow.tracking import MlflowClient
from mlflow.entities import Experiment
from mlflow.models import infer_signature

from src.pipelines import DatasetBuilderPipeline
from settings.dataset_builder_pipeline_settings import DatasetBuilderPipelineSettings
from settings import MLFLOW_SETTINGS, GUITAR_SET_SETTINGS

warnings.filterwarnings("ignore")

RANDOM_STATE = 73
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

c:\Users\Administrateur\Documents\M2i_CDSD_Projet\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\Administrateur\Documents\M2i_CDSD_Projet\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
os.environ["AWS_ACCESS_KEY_ID"] = MLFLOW_SETTINGS.aws_access_key_id
os.environ["AWS_SECRET_ACCESS_KEY"] = MLFLOW_SETTINGS.aws_secret_access_key
os.environ["MLFLOW_S3_ENDPOINT_URL"] = MLFLOW_SETTINGS.s3_endpoint_url
os.environ["AWS_REGION"] = MLFLOW_SETTINGS.aws_region

## Configuration MLflow

In [6]:
def get_or_restore_experiment(experiment_name: str) -> Experiment:
    client = MlflowClient()

    experiment = client.get_experiment_by_name(experiment_name)

    if experiment is None:
        client.create_experiment(experiment_name)
        return client.get_experiment_by_name(experiment_name)

    if experiment.lifecycle_stage == "deleted":
        client.restore_experiment(experiment.experiment_id)

    return experiment

In [7]:
MLFLOW_EXPERIMENT_NAME = "cqt_rcnn"

mlflow.set_tracking_uri(MLFLOW_SETTINGS.tracking_uri)

experiment = get_or_restore_experiment(MLFLOW_EXPERIMENT_NAME)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print("MLflow tracking URI :", mlflow.get_tracking_uri())
print("Experiment ID       :", experiment.experiment_id)
print("Experiment name     :", experiment.name)
print("Lifecycle stage     :", experiment.lifecycle_stage)

MLflow tracking URI : http://localhost:5000
Experiment ID       : 2
Experiment name     : cqt_rcnn
Lifecycle stage     : active


## Chargement des données

In [8]:
settings_context_window = DatasetBuilderPipelineSettings()
settings_context_window.output_dataset_name = "guitar_set_context_window_11"
settings_context_window.datasets_used = (GUITAR_SET_SETTINGS.name,)
settings_context_window.preprocessing_pipeline_id = None
settings_context_window.train_size = 0.7
settings_context_window.validation_size = 0.1
settings_context_window.test_size = 0.2
settings_context_window.random_state = 73
settings_context_window.shuffle = True
settings_context_window.use_context_window = True
settings_context_window.context_size = 11

dataset_builder_pipeline = DatasetBuilderPipeline(
    logging.getLogger(), settings=settings_context_window
)

train_dataset, validation_dataset, test_dataset = dataset_builder_pipeline.run()

In [9]:
train_features, train_target = train_dataset
X_train = train_features.values
y_train = train_target.values

validation_features, validation_target = validation_dataset
X_validation = validation_features.values
y_validation = validation_target.values

test_features, test_target = test_dataset
X_test = test_features.values
y_test = test_target.values

feature_names = train_features.columns.to_list()
target_names = train_target.columns.to_list()

print(
    f"Dimension jeu d'entrainement : features={X_train.shape}, target={y_train.shape}"
)
print(
    f"Dimension jeu de validation  : features={X_validation.shape}, target={y_validation.shape}"
)
print(f"Dimension jeu de test        : features={X_test.shape}, target={y_test.shape}")
print()

print("Noms des features :", feature_names)
print("Noms des targets  :", target_names)
print()

Dimension jeu d'entrainement : features=(342529, 1932), target=(342529, 49)
Dimension jeu de validation  : features=(38591, 1932), target=(38591, 49)
Dimension jeu de test        : features=(91440, 1932), target=(91440, 49)

Noms des features : ['cqt_0_t-11', 'cqt_1_t-11', 'cqt_2_t-11', 'cqt_3_t-11', 'cqt_4_t-11', 'cqt_5_t-11', 'cqt_6_t-11', 'cqt_7_t-11', 'cqt_8_t-11', 'cqt_9_t-11', 'cqt_10_t-11', 'cqt_11_t-11', 'cqt_12_t-11', 'cqt_13_t-11', 'cqt_14_t-11', 'cqt_15_t-11', 'cqt_16_t-11', 'cqt_17_t-11', 'cqt_18_t-11', 'cqt_19_t-11', 'cqt_20_t-11', 'cqt_21_t-11', 'cqt_22_t-11', 'cqt_23_t-11', 'cqt_24_t-11', 'cqt_25_t-11', 'cqt_26_t-11', 'cqt_27_t-11', 'cqt_28_t-11', 'cqt_29_t-11', 'cqt_30_t-11', 'cqt_31_t-11', 'cqt_32_t-11', 'cqt_33_t-11', 'cqt_34_t-11', 'cqt_35_t-11', 'cqt_36_t-11', 'cqt_37_t-11', 'cqt_38_t-11', 'cqt_39_t-11', 'cqt_40_t-11', 'cqt_41_t-11', 'cqt_42_t-11', 'cqt_43_t-11', 'cqt_44_t-11', 'cqt_45_t-11', 'cqt_46_t-11', 'cqt_47_t-11', 'cqt_48_t-11', 'cqt_49_t-11', 'cqt_50_t-11',

## Normalisation des données

In [10]:
def normalize(data: np.ndarray) -> np.ndarray:
    return (data + 80) / 80


X_train_normalized = normalize(X_train)
X_validation_normalized = normalize(X_validation)
X_test_normalized = normalize(X_test)

In [11]:
window_size = 2 * settings_context_window.context_size + 1
n_features = X_train_normalized.shape[1] // window_size

print("Taille de la fenêtre de contexte :", window_size)
print("Nombre de features               :", n_features)

Taille de la fenêtre de contexte : 23
Nombre de features               : 84


In [12]:
X_train_reshaped = X_train_normalized.reshape(-1, window_size, n_features, 1)
X_validation_reshaped = X_validation_normalized.reshape(-1, window_size, n_features, 1)
X_test_reshaped = X_test_normalized.reshape(-1, window_size, n_features, 1)

print(
    f"Dimension jeu d'entrainement reformé : features={X_train_reshaped.shape}, target={y_train.shape}"
)
print(
    f"Dimension jeu de validation reformé  : features={X_validation_reshaped.shape}, target={y_validation.shape}"
)
print(
    f"Dimension jeu de test reformé        : features={X_test_reshaped.shape}, target={y_test.shape}"
)
print()

Dimension jeu d'entrainement reformé : features=(342529, 23, 84, 1), target=(342529, 49)
Dimension jeu de validation reformé  : features=(38591, 23, 84, 1), target=(38591, 49)
Dimension jeu de test reformé        : features=(91440, 23, 84, 1), target=(91440, 49)



## Définition des modèles

### Callbacks

In [13]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1,
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1,
)

### CNN + MLP

In [14]:
input_dim = X_train_reshaped.shape[1:]
output_dim = y_train.shape[1]
print("Imput dimension  :", input_dim)
print("Output dimension :", output_dim)

Imput dimension  : (23, 84, 1)
Output dimension : 49


In [15]:
def build_cnn_mlp_model(
    input_dim: tuple[int, int, int],
    output_dim: int,
    optimizer: tf.keras.optimizers.Optimizer,
    activation_layer: tf.keras.layers.Layer | None = None,
    use_batch_norm: bool = True,
    conv_filters: list[int] = [64, 128, 256],
    kernel_sizes: list[tuple[int, int]] = [(3, 3), (3, 5), (3, 7)],
    convs_per_block: int = 2,
    pool_size: tuple[int, int] = (2, 2),
    hidden_units: list[int] = [512, 256, 128],
    dropout_rates: list[float] | None = [0.3, 0.2, 0.1],
    weight_decay: float = 1e-4,
):

    if activation_layer is None:
        activation_layer = tf.keras.layers.ELU()

    if len(kernel_sizes) != len(conv_filters):
        raise ValueError("'kernel_sizes' and 'conv_filters' must have the same length")

    if dropout_rates is None:
        dropout_rates = [0.0] * len(hidden_units)

    if len(dropout_rates) != len(hidden_units):
        raise ValueError("'dropout_rates' and 'hidden_units' must have the same length")

    he_init = tf.keras.initializers.HeNormal()

    inputs = tf.keras.Input(shape=input_dim)

    x = inputs

    # ===
    # CNN
    # ===

    for filters, kernel_size in zip(conv_filters, kernel_sizes):
        for _ in range(convs_per_block):
            x = tf.keras.layers.Conv2D(
                filters=filters,
                kernel_size=kernel_size,
                padding="same",
                kernel_initializer=he_init,
                kernel_regularizer=tf.keras.regularizers.l2(weight_decay),
                use_bias=not use_batch_norm,
            )(x)

            if use_batch_norm:
                x = tf.keras.layers.BatchNormalization()(x)

            x = activation_layer(x)

        x = tf.keras.layers.MaxPooling2D(pool_size=pool_size)(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    # ===
    # MLP
    # ===

    for units, dropout_rate in zip(hidden_units, dropout_rates):
        x = tf.keras.layers.Dense(
            units,
            kernel_initializer=he_init,
            kernel_regularizer=tf.keras.regularizers.l2(weight_decay),
            use_bias=not use_batch_norm,
        )(x)

        if use_batch_norm:
            x = tf.keras.layers.BatchNormalization()(x)

        x = activation_layer(x)

        if dropout_rate > 0:
            x = tf.keras.layers.Dropout(dropout_rate)(x)

    outputs = tf.keras.layers.Dense(output_dim, activation="sigmoid")(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.F1Score(
                average="micro",
                threshold=0.5,
                name="f1_micro",
            ),
        ],
    )

    return model


build_cnn_mlp_light_model = functools.partial(
    build_cnn_mlp_model,
    input_dim=input_dim,
    output_dim=output_dim,
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=3e-4,
        weight_decay=1e-4,
    ),
    activation_layer=tf.keras.layers.ELU(),
    use_batch_norm=True,
    conv_filters=[32, 64],
    kernel_sizes=[(3, 3), (3, 3)],
    convs_per_block=1,
    pool_size=(2, 2),
    hidden_units=[256],
    dropout_rates=[0.3],
    weight_decay=1e-4,
)

build_cnn_mlp_balanced_model = functools.partial(
    build_cnn_mlp_model,
    input_dim=input_dim,
    output_dim=output_dim,
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=3e-4,
        weight_decay=1e-4,
    ),
    activation_layer=tf.keras.layers.ELU(),
    use_batch_norm=True,
    conv_filters=[64, 128, 256],
    kernel_sizes=[(3, 3), (3, 5), (3, 7)],
    convs_per_block=2,
    pool_size=(2, 2),
    hidden_units=[512, 256, 128],
    dropout_rates=[0.3, 0.2, 0.1],
    weight_decay=1e-4,
)

build_cnn_mlp_frequency_model = functools.partial(
    build_cnn_mlp_model,
    input_dim=input_dim,
    output_dim=output_dim,
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=3e-4,
        weight_decay=1e-4,
    ),
    activation_layer=tf.keras.layers.ELU(),
    use_batch_norm=True,
    conv_filters=[64, 128, 256],
    kernel_sizes=[(3, 5), (3, 7), (3, 9)],
    convs_per_block=2,
    pool_size=(2, 2),
    hidden_units=[512, 256],
    dropout_rates=[0.3, 0.3],
    weight_decay=1e-4,
)

### RCNN (CNN + BiLSTM)

In [16]:
def build_rcnn_model(
    input_dim: tuple[int, int, int],
    output_dim: int,
    optimizer: tf.keras.optimizers.Optimizer,
    activation_layer: tf.keras.layers.Layer | None = None,
    use_batch_norm: bool = True,
    conv_filters: list[int] = [64, 128, 256],
    kernel_sizes: list[tuple[int, int]] = [(3, 3), (3, 5), (3, 7)],
    convs_per_block: int = 2,
    pool_size: tuple[int, int] = (1, 2),
    lstm_units: int = 128,
    lstm_dropout: float = 0.3,
    dense_units: list[int] = [256],
    dense_dropout_rates: list[float] = [0.3],
    weight_decay=1e-4,
):

    if activation_layer is None:
        activation_layer = tf.keras.layers.ELU()

    if len(kernel_sizes) != len(conv_filters):
        raise ValueError("'kernel_sizes' and 'conv_filters' must have the same length")

    if dense_dropout_rates is None:
        dense_dropout_rates = [0.0] * len(dense_units)

    if len(dense_dropout_rates) != len(dense_units):
        raise ValueError(
            "'dense_dropout_rates' and 'dense_units' must have the same length"
        )

    he_init = tf.keras.initializers.HeNormal()

    inputs = tf.keras.Input(shape=input_dim)

    x = inputs

    # ===
    # CNN
    # ===

    for filters, kernel_size in zip(conv_filters, kernel_sizes):
        for _ in range(convs_per_block):
            x = tf.keras.layers.Conv2D(
                filters=filters,
                kernel_size=kernel_size,
                padding="same",
                kernel_initializer=he_init,
                kernel_regularizer=tf.keras.regularizers.l2(weight_decay),
                use_bias=not use_batch_norm,
            )(x)

            if use_batch_norm:
                x = tf.keras.layers.BatchNormalization()(x)

            x = activation_layer(x)

        x = tf.keras.layers.MaxPooling2D(pool_size=pool_size)(x)

    x = tf.keras.layers.TimeDistributed(tf.keras.layers.Flatten())(x)

    # ===
    # Bidirectional LSTM
    # ===

    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(
            lstm_units,
            return_sequences=False,
            dropout=lstm_dropout,
        )
    )(x)

    # ===
    # MLP
    # ===

    for units, dropout_rate in zip(dense_units, dense_dropout_rates):
        x = tf.keras.layers.Dense(
            units,
            kernel_initializer=he_init,
            kernel_regularizer=tf.keras.regularizers.l2(weight_decay),
            use_bias=not use_batch_norm,
        )(x)

        if use_batch_norm:
            x = tf.keras.layers.BatchNormalization()(x)

        x = activation_layer(x)

        if dropout_rate > 0:
            x = tf.keras.layers.Dropout(dropout_rate)(x)

    outputs = tf.keras.layers.Dense(output_dim, activation="sigmoid")(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.F1Score(
                average="micro",
                threshold=0.5,
                name="f1_micro",
            ),
        ],
    )

    return model


build_rcnn_light_model = functools.partial(
    build_rcnn_model,
    input_dim=input_dim,
    output_dim=output_dim,
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=3e-4,
        weight_decay=1e-4,
    ),
    activation_layer=tf.keras.layers.ELU(),
    use_batch_norm=True,
    conv_filters=[64, 128],
    kernel_sizes=[(3, 3), (3, 5)],
    convs_per_block=1,
    pool_size=(1, 2),
    lstm_units=64,
    lstm_dropout=0.3,
    dense_units=[128],
    dense_dropout_rates=[0.3],
)

build_rcnn_balanced_model = functools.partial(
    build_rcnn_model,
    input_dim=input_dim,
    output_dim=output_dim,
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=3e-4,
        weight_decay=1e-4,
    ),
    activation_layer=tf.keras.layers.ELU(),
    use_batch_norm=True,
    conv_filters=[64, 128, 256],
    kernel_sizes=[(3, 3), (3, 5), (3, 7)],
    convs_per_block=2,
    pool_size=(1, 2),
    lstm_units=128,
    lstm_dropout=0.3,
    dense_units=[256],
    dense_dropout_rates=[0.3],
)

build_rcnn_frequency_model = functools.partial(
    build_rcnn_model,
    input_dim=input_dim,
    output_dim=output_dim,
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=3e-4,
        weight_decay=1e-4,
    ),
    activation_layer=tf.keras.layers.ELU(),
    use_batch_norm=True,
    conv_filters=[64, 128, 256],
    kernel_sizes=[(3, 5), (3, 7), (3, 9)],
    convs_per_block=2,
    pool_size=(1, 2),
    lstm_units=128,
    lstm_dropout=0.3,
    dense_units=[256],
    dense_dropout_rates=[0.3],
)

## Evaluation des modèles

La transcription audio → MIDI est formulée comme un problème de classification multi-label frame-wise :

- chaque ligne correspond à une frame temporelle ;
- chaque colonne correspond à une note MIDI ;
- plusieurs notes peuvent être actives simultanément.

Exemple :

| Frame | C4 | D4 | E4 | F4 |
|---------|----|----|----|----|
| t₁ | 1 | 0 | 1 | 0 |
| t₂ | 0 | 0 | 1 | 1 |

Une erreur peut donc être commise :
- sur une note spécifique (par exemple une note non détectée),
- sur une frame complète (par exemple une frame non parfaitement transcrite)
- sur la structure musicale globale (par exemple une note transcrite discontinuement qui devrait être continue).

Nous utilisons donc plusieurs métriques complémentaires pour capturer tous ces aspects.

### F1-score Micro

**Définition :** Le F1-score est la moyenne harmonique entre la précision et le rappel.
Dans le cas **micro**, tous les labels de toutes les frames sont regroupés avant calcul.

$$
Precision_{micro}
=
\frac{\sum TP}
{\sum TP + \sum FP}
$$

$$
Recall_{micro}
=
\frac{\sum TP}
{\sum TP + \sum FN}
$$

$$
F1_{micro}
=
2 \cdot
\frac{
Precision_{micro}
\cdot
Recall_{micro}
}
{
Precision_{micro}
+
Recall_{micro}
}
$$

**Utilité :** Cette métrique répond à la question : "Quelle est la qualité globale de la transcription ?"
Toutes les prédictions sont considérées ensemble, c'est à dire, toutes les notes, toutes les frames, tous les morceaux.

**Interprétation :**

| Valeur | Interprétation |
| :- | :- |
| 1.0 | transcription parfaite |
| > 0.9 | excellente |
| 0.8 - 0.9 | très bonne |
| 0.7 - 0.8 | correcte |
| < 0.7 | amélioration nécessaire |

Le F1 micro constitue la métrique principale pour comparer plusieurs modèles.

### F1-score Macro

**Définition :** On calcule d'abord un F1-score pour chaque note MIDI $F1_k$, puis on effectue la moyenne :

$$
F1_{macro}
=
\frac{1}{K}
\sum_{k=1}^{K}
F1_k
$$

où $k$ représente le nombre total de notes MIDI modélisées.

**Utilité :** Le F1 micro est dominé par les notes les plus fréquentes.
Le F1 macro donne le même poids à une note très fréquente et à une note très rare.
Il permet donc d'évaluer la capacité du modèle à généraliser sur l'ensemble du registre de la guitare.

**Interprétation :**
Un écart important entre $F1_{micro} \gg F1_{macro}$ indique généralement que les notes fréquentes sont bien reconnues et que les notes rares sont mal reconnues. Ce peut être le signe d'un déséquilibre de classes.

### Precision

**Définition :**

$$
Precision = \frac{TP}{TP + FP}
$$

où TP signifie True Positives et FP signifie False Positives.

**Utilité :** La précision répond à la question : "Quand le modèle prédit une note, a-t-il raison ?". Une faible précision signifie que le modèle ajoute beaucoup de notes inexistantes.

**Interprétation :**
Une précision faible révèle un grand nombre de notes inexistantes et une transcription surchargée.
Une précision élevée indique qu'il y a peu de fausses notes et que la transcription est propre.

### Recall

**Définition :**

$$
Recall = \frac{TP}{TP + FN}
$$

où TP signifie True Positives et FN signifie False Negatives

**Utilité :** Le rappel répond à la question : "Combien de vraies notes le modèle retrouve-t-il ?"

**Interprétation :**
Un recall faible révèle que le modèle oublie des notes et que transcription incomplète.
Un recall élevé montre que davantage de notes sont détectées, parfois au prix de faux positifs supplémentaires.

### Hamming Loss

**Définition :**

$$
HammingLoss = \frac{FP + FN}{N \times K}
$$

avec $N$ le nombre de frames et $K$ le nombre de notes MIDI.

**Utilité :** Cette métrique mesure le taux d'erreur moyen par note et par frame.
Contrairement au F1-score, elle pénalise directement chaque erreur élémentaire.


**Interprétation :**

| Valeur | Signification |
| :- | :- |
| 0 | aucune erreur |
| 0.01 | 1 % d'erreurs |
| 0.05 | 5 % d'erreurs |
| 0.10 | 10 % d'erreurs |

Plus la valeur est faible, meilleur est le modèle.

### Subset Accuracy

**Définition :** Une frame est correcte uniquement si toutes les notes sont correctement prédites.

$$
SubsetAccuracy = \frac{\#\;frames\;parfaites}{\#\;frames}
$$

**Utilité :** Cette métrique est extrêmement stricte.
Elle répond à la question : "Combien de frames sont parfaitement transcrites ?"

**Interprétation :**
Même un très bon modèle obtient souvent une valeur relativement faible.
Cette métrique permet de mesurer la qualité des accords complets.

In [17]:
def compute_ml_metrics(y_true, y_pred):
    return {
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_micro": precision_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "recall_micro": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "subset_accuracy": accuracy_score(y_true, y_pred),
    }

### F1-score par pitch MIDI

**Définition :** Pour chaque note MIDI :

$$
F1_k = 2 \cdot \frac{Precision_k \cdot Recall_k}{Precision_k + Recall_k}
$$

**Utilité :** Le score global peut masquer des difficultés spécifiques.
Certaines notes peuvent être très bien reconnues et d'autres très mal reconnues.

Le F1 par pitch permet d'identifier les zones du manche difficiles, les fréquences mal représentées ou les erreurs de feature engineering.

**Interprétation :** Un graphique F1 par pitch permet de visualiser les notes problématiques, les tendances graves / aigus et les limites du modèle.

In [18]:
def compute_f1_per_pitch(
    y_true,
    y_pred,
    pitch_offset=40,  # /!\ Regarder les settings de la pipeline de prétraitement
):
    scores = []

    for k in range(y_true.shape[1]):
        scores.append(
            {
                "pitch_midi": k + pitch_offset,
                "f1_score": f1_score(
                    y_true[:, k],
                    y_pred[:, k],
                    average="binary",
                    zero_division=0,
                ),
            }
        )

    return pd.DataFrame(scores)

### Pitch Tolerance Accuracy

**Définition :** Une prédiction est considérée correcte si elle est proche de la vraie note :

$$
|Pitch_{pred} - Pitch_{true}| \leq t
$$

où $t$ représente le nombre de demi-tons.

**Utilité :** Une erreur d'un demi-ton est moins grave musicalement qu'une erreur d'une octave.
Le F1-score classique considère pourtant ces deux erreurs comme identiques.
Cette métrique introduit une notion de proximité musicale.

**Interprétation :** Une pitch tolerance élevée indique que le modèle comprend globalement les hauteurs de notes.
Une pitch tolerance faible indique que le modèle commet des erreurs importantes sur les hauteurs de notes.

In [19]:
def pitch_tolerance_accuracy(y_true, y_pred, tolerance=1):
    true_idx = np.where(y_true == 1)
    pred_idx = np.where(y_pred == 1)

    if len(true_idx[0]) == 0:
        return 0.0

    correct = 0

    for i in range(len(true_idx[0])):
        t_frame = true_idx[0][i]
        t_pitch = true_idx[1][i]

        frame_preds = pred_idx[1][pred_idx[0] == t_frame]

        if len(frame_preds) == 0:
            continue

        if np.any(np.abs(frame_preds - t_pitch) <= tolerance):
            correct += 1

    return correct / len(true_idx[0])

### Activation Ratio

**Définition :**
$$
ActivationRatio = \frac{\text{taux d'activation prédit}}{\text{taux d'activation réel}}
$$

**Utilité :** Cette métrique mesure le biais global du modèle.

**Interprétation :**
- $Ratio \approx 1$ : Le modèle produit globalement le bon nombre de notes.
- $Ratio > 1$ : Le modèle sur-prédit, il ajoute trop de notes.
- $Ratio < 1$ : Le modèle sous-prédit, il manque des notes.

In [20]:
def activation_ratio(y_true, y_pred):
    return {
        "true_activation": y_true.mean(),
        "pred_activation": y_pred.mean(),
        "ratio": (y_pred.mean() / (y_true.mean() + 1e-8)),
    }

### Temporal Jitter

**Définition :** Le jitter mesure les variations de prédictions entre frames successives.
Une approximation simple est :

$$
Jitter = mean \left(|y_t - y_{t-1}| \right)
$$

**Utilité :** La transcription frame-wise produit souvent un phénomène appelé *flickering*.
Une note apparaît puis disparaît très rapidement alors qu'elle devrait rester stable.

**Interprétation :**
Un jitter faible indique une transcription stable avec des notes continues.
Un jitter élevé révèle une instabilité temporelle.

In [21]:
def temporal_jitter(y_pred):
    return np.mean(np.abs(np.diff(y_pred, axis=0)))

### Pitch Class Confusion Matrix

**Définition :**
Une note MIDI peut être ramenée à sa classe de hauteur (Pitch Class) : $PitchClass = MIDI \bmod 12$
Les notes séparées d'une ou plusieurs octaves appartiennent donc à la même classe.

**Utilité :** La Pitch Class Confusion Matrix regroupe les notes par nom musical (C, C#, D, D#, E, F, F#, G, G#, A, A#, B).
Elle permet de mettre en évidence des erreurs harmoniques ou tonales.
Deux erreurs peuvent avoir le même impact sur le F1-score, par exemple, prédire E au lieu de F et prédire E au lieu de A#.
Pourtant musicalement, ces erreurs sont très différentes.
La Pitch Class Confusion Matrix permet d'analyser la nature musicale des erreurs plutôt que leur simple quantité.

**Interprétation :**
Une diagonale dominante indique que les classes de hauteur sont correctement reconnues.
Des valeurs importantes hors diagonale indiquent des confusions entre notes voisines, des difficultés dans certaines régions fréquentielles et d'éventuels problèmes liés aux harmoniques de la guitare.

In [22]:
def pitch_class_confusion(y_true, y_pred):
    true_pc = np.where(y_true == 1)[1] % 12
    pred_pc = np.where(y_pred == 1)[1] % 12

    cm = np.zeros((12, 12))

    for t, p in zip(true_pc, pred_pc):
        cm[t, p] += 1

    return cm

In [23]:
def evaluate(
    model,
    X_test,
    y_test,
    label_names,
    pitch_offset=40,  # /!\ Regarder les settings de la pipeline de prétraitement
    threshold=0.5,
):
    y_score = model.predict(X_test, verbose=0)

    y_pred = (y_score >= threshold).astype(np.int32)

    report = classification_report(
        y_test,
        y_pred,
        target_names=label_names,
        output_dict=True,
        zero_division=0,
    )

    metrics = {}
    metrics.update(compute_ml_metrics(y_test, y_pred))

    metrics["pitch_acc_tol_1"] = pitch_tolerance_accuracy(y_test, y_pred, tolerance=1)
    metrics["pitch_acc_tol_2"] = pitch_tolerance_accuracy(y_test, y_pred, tolerance=2)

    metrics.update(activation_ratio(y_test, y_pred))

    metrics["jitter"] = temporal_jitter(y_pred)

    df_f1_per_pitch = compute_f1_per_pitch(y_test, y_pred, pitch_offset)

    cm = confusion_matrix(y_test.flatten(), y_pred.flatten())

    pitch_class_cm = pitch_class_confusion(y_test, y_pred)

    artifacts = {
        "y_pred": y_pred,
        "y_score": y_score,
        "confusion_matrix": cm,
        "classification_report": report,
        "f1_per_pitch": df_f1_per_pitch,
        "pitch_class_confusion_matrix": pitch_class_cm,
    }

    return metrics, artifacts

In [24]:
def log_confusion_matrix(cm, artifact_file="confusion_matrix.png"):
    cm_percent = cm / cm.sum().sum() * 100

    plt.figure(figsize=(7, 5))

    sns.heatmap(
        cm_percent,
        annot=True,
        fmt=".2f",
        cmap="cividis",
        square=True,
        linewidths=0.6,
        linecolor="white",
        annot_kws={"size": 10},
    )

    plt.title("Matrice de confusion")
    plt.xlabel("Predict label")
    plt.ylabel("True label")

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

In [25]:
def log_f1_per_pitch(df_scores, artifact_file="f1_per_pitch.png"):
    plt.figure(figsize=(10, 4))

    plt.plot(
        df_scores["pitch_midi"],
        df_scores["f1_score"],
    )

    plt.title("F1-score per Pitch")
    plt.xlabel("MIDI Pitch")
    plt.ylabel("F1-score")

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

In [26]:
def log_precision_recall_curve(
    y_true, y_score, artifact_file="precision_recall_curve.png"
):
    precision, recall, _ = precision_recall_curve(
        y_true.flatten(),
        y_score.flatten(),
    )

    plt.figure(figsize=(6, 6))

    plt.plot(recall, precision)

    plt.title("Global Precision-Recall Curve")
    plt.xlabel("Recall")
    plt.ylabel("Precision")

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

### Training History

**Principe :**
À chaque époque d'entraînement, on mesure les performances du modèle sur le jeu d'entraînement et sur le jeu de validation. Les courbes retracent l'évolution des métriques (loss, précision, rappel, F1, etc.) au fil des époques.

**Utilité :**
L'historique d'entraînement permet de répondre à plusieurs questions :

* le modèle continue-t-il à apprendre ?
* l'entraînement a-t-il convergé ?
* le modèle commence-t-il à sur-apprendre ?
* les callbacks (Early Stopping, ReduceLROnPlateau) interviennent-ils au bon moment ?
* davantage d'époques seraient-elles bénéfiques ?

**Interprétation :**

* Loss entraînement diminue et loss validation diminue : apprentissage sain.
* Loss entraînement diminue mais loss validation augmente : sur-apprentissage.
* Loss entraînement et validation stagnent à des valeurs élevées : sous-apprentissage.
* Les métriques de validation continuent à s'améliorer en fin d'entraînement : davantage d'époques pourraient améliorer les performances.
* Les métriques de validation se stabilisent tandis que les métriques d'entraînement continuent à progresser : le modèle atteint probablement sa capacité de généralisation maximale.
* Une forte instabilité des métriques entre époques peut révéler un taux d'apprentissage trop élevé, un batch size trop faible ou un jeu de données insuffisant.

In [27]:
def log_training_history(
    history,
    metric="f1_micro",
    loss="binary_crossentropy",
    artifact_file="training_history.png",
):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Loss
    ax1.plot(history.history["loss"], label="Train loss")
    ax1.plot(history.history["val_loss"], label="Validation loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel(f"Loss ({loss.upper()})")
    ax1.set_title(f"Loss evolution ({loss.upper()})")
    ax1.legend()

    # Metric
    ax2.plot(history.history[metric], label=f"Train {metric.upper()}")
    ax2.plot(history.history[f"val_{metric}"], label=f"Validation {metric.upper()}")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel(f"{metric.upper()}")
    ax2.set_title(f"{metric.upper()} evolution")
    ax2.legend()

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

## Expériences

In [28]:
def optimize_threshold(
    model,
    X_validation,
    y_validation,
    thresholds=np.arange(0.05, 1.0, 0.05),
):
    y_score = model.predict(X_validation, verbose=0)

    results = []

    for threshold in thresholds:
        y_pred = (y_score >= threshold).astype(np.uint8)

        results.append(
            {
                "threshold": threshold,
                "f1_micro": f1_score(
                    y_validation,
                    y_pred,
                    average="micro",
                    zero_division=0,
                ),
                "f1_macro": f1_score(
                    y_validation,
                    y_pred,
                    average="macro",
                    zero_division=0,
                ),
            }
        )

    results = pd.DataFrame(results)

    best_idx = results["f1_micro"].idxmax()

    return (
        float(results.loc[best_idx, "threshold"]),
        float(results.loc[best_idx, "f1_micro"]),
        results,
    )

In [29]:
def log_threshold_search(results: pd.DataFrame, artifact_file="threshold_search.png"):
    fig, ax = plt.subplots(figsize=(7, 5))

    ax.plot(
        results["threshold"],
        results["f1_micro"],
        marker="x",
    )

    ax.set_xlabel("Threshold")
    ax.set_ylabel("F1 score")
    ax.set_title("Threshold optimization")

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

In [30]:
def run_experiment(model_factory, run_name, tags):

    print("=" * 80)
    print(f"[{datetime.now()}] Starting run: {run_name}")

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id
        print(f"[{datetime.now()}] MLflow run_id: {run_id}")

        mlflow.set_tags(tags)

        print(f"[{datetime.now()}] Building model...")
        model = model_factory()

        print(f"[{datetime.now()}] Logging dataset name and id...")
        mlflow.log_params(
            {"dataset_id": dataset_builder_pipeline.pipeline_metadata["_id"]}
        )
        mlflow.log_params({"dataset_name": settings_context_window.output_dataset_name})

        print(f"[{datetime.now()}] Logging model config...")
        config_path = ARTIFACT_DIR / "model_config.json"
        with open(config_path, "w") as f:
            json.dump(model.get_config(), f)
        mlflow.log_artifact(str(config_path))

        print(f"[{datetime.now()}] Training model...")
        t0 = perf_counter()
        history = model.fit(
            X_train_reshaped,
            y_train,
            validation_data=(X_validation_reshaped, y_validation),
            batch_size=32,
            epochs=200,
            callbacks=[early_stopping, reduce_lr],
            verbose=1,
        )
        fitting_time = perf_counter() - t0
        mlflow.log_metric("fitting_time", fitting_time)
        print(f"[{datetime.now()}] Training completed ({fitting_time:.1f}s)")

        print(f"[{datetime.now()}] Logging best validation score...")
        best_epoch = np.argmax(history.history["val_f1_micro"])
        mlflow.log_metric(
            "best_validation_f1_micro", history.history["val_f1_micro"][best_epoch]
        )
        mlflow.log_metric(
            "best_validation_precision", history.history["val_precision"][best_epoch]
        )
        mlflow.log_metric(
            "best_validation_recall", history.history["val_recall"][best_epoch]
        )

        print(f"[{datetime.now()}] Searching optimal threshold...")
        best_threshold, validation_score, threshold_results = optimize_threshold(
            model,
            X_validation_reshaped,
            y_validation,
        )
        print(
            f"[{datetime.now()}] Best threshold : {best_threshold} (validation_f1_micro={validation_score:.4f})"
        )

        mlflow.log_param("prediction_threshold", best_threshold)
        mlflow.log_metric("validation_f1_micro_best_threshold", validation_score)

        print(f"[{datetime.now()}] Saving threshold search...")
        log_threshold_search(threshold_results)
        threshold_csv_path = ARTIFACT_DIR / "threshold_search.csv"
        threshold_results.to_csv(threshold_csv_path, index=False)
        mlflow.log_artifact(str(threshold_csv_path))

        print(f"[{datetime.now()}] Evaluating on test set...")
        metrics, artifacts = evaluate(
            model=model,
            X_test=X_test_reshaped,
            y_test=y_test,
            label_names=target_names,
            threshold=best_threshold,
        )
        print(f"[{datetime.now()}] Evaluation completed ({len(metrics)} metrics)")

        print(f"[{datetime.now()}] Logging metrics...")
        mlflow.log_metrics(metrics)

        print(f"[{datetime.now()}] Saving classification report...")
        report_path = ARTIFACT_DIR / "classification_report.json"
        with open(report_path, "w") as f:
            json.dump(
                artifacts["classification_report"],
                f,
                indent=2,
            )
        mlflow.log_artifact(str(report_path))

        print(f"[{datetime.now()}] Saving pitch metrics...")
        f1_pitch_csv_path = ARTIFACT_DIR / "f1_per_pitch.csv"
        artifacts["f1_per_pitch"].to_csv(
            f1_pitch_csv_path,
            index=False,
        )
        mlflow.log_artifact(str(f1_pitch_csv_path))

        print(f"[{datetime.now()}] Logging confusion matrix...")
        log_confusion_matrix(artifacts["confusion_matrix"])

        print(f"[{datetime.now()}] Logging F1-per-pitch plot...")
        log_f1_per_pitch(artifacts["f1_per_pitch"])

        print(f"[{datetime.now()}] Logging precision-recall curve...")
        log_precision_recall_curve(y_test, artifacts["y_score"])

        print(f"[{datetime.now()}] Logging training history...")
        log_training_history(history)

        print(f"[{datetime.now()}] Logging tensorflow model...")
        input_example = X_train_reshaped[:5]
        prediction = model.predict(input_example)
        signature = infer_signature(input_example, prediction)
        mlflow.tensorflow.log_model(
            model=model,
            name="model",
            signature=signature,
            input_example=input_example,
        )
        print(f"[{datetime.now()}] Model logged successfully")

        print("=" * 80)
        print("Run completed")
        print(f"Run ID : {run_id}")

        print("\nMain metrics:")

        summary_metrics = [
            "test_f1_micro",
            "test_f1_macro",
            "test_precision_micro",
            "test_recall_micro",
        ]

        for metric in summary_metrics:
            if metric in metrics:
                print(f"{metric}: {metrics[metric]:.4f}")

        print("=" * 80)

In [ ]:
experiments = [
    # (
    #     build_cnn_mlp_light_model,
    #     "cnn_mlp_light_context_window",
    #     {
    #         "task": "audio_to_midi",
    #         "representation": "context_window",
    #         "model_family": "tensorflow",
    #         "model": "cnn_mlp_light",
    #     },
    # ),
    # (
    #     build_cnn_mlp_balanced_model,
    #     "cnn_mlp_balanced_context_window",
    #     {
    #         "task": "audio_to_midi",
    #         "representation": "context_window",
    #         "model_family": "tensorflow",
    #         "model": "cnn_mlp_balanced",
    #     },
    # ),
    # (
    #     build_cnn_mlp_frequency_model,
    #     "cnn_mlp_frequency_context_window",
    #     {
    #         "task": "audio_to_midi",
    #         "representation": "context_window",
    #         "model_family": "tensorflow",
    #         "model": "cnn_mlp_frequency",
    #     },
    # ),
    # (
    #     build_rcnn_light_model,
    #     "rcnn_light_context_window",
    #     {
    #         "task": "audio_to_midi",
    #         "representation": "context_window",
    #         "model_family": "tensorflow",
    #         "model": "rcnn_light",
    #     },
    # ),
    # (
    #     build_rcnn_balanced_model,
    #     "rcnn_balanced_context_window",
    #     {
    #         "task": "audio_to_midi",
    #         "representation": "context_window",
    #         "model_family": "tensorflow",
    #         "model": "rcnn_balanced",
    #     },
    # ),
    (
        build_rcnn_frequency_model,
        "rcnn_frequency_context_window",
        {
            "task": "audio_to_midi",
            "representation": "context_window",
            "model_family": "tensorflow",
            "model": "rcnn_frequency",
        },
    ),
]

for model_factory, run_name, tags in experiments:
    run_experiment(model_factory, run_name, tags)

[2026-07-09 21:10:54.946239] Starting run: rcnn_frequency_context_window
[2026-07-09 21:10:55.140206] MLflow run_id: 04afdc9903af4f5c89607f2cf3c1051b
[2026-07-09 21:10:55.157037] Building model...
[2026-07-09 21:10:55.293430] Logging dataset name and id...
[2026-07-09 21:10:55.413002] Logging model config...
[2026-07-09 21:10:55.797728] Training model...
Epoch 1/200
10705/10705 ━━━━━━━━━━━━━━━━━━━━ 6817s 636ms/step - f1_micro: 0.7668 - loss: 0.0899 - precision: 0.7877 - recall: 0.7469 - val_f1_micro: 0.8771 - val_loss: 0.0343 - val_precision: 0.8780 - val_recall: 0.8762 - learning_rate: 3.0000e-04
Epoch 2/200
 7815/10705 ━━━━━━━━━━━━━━━━━━━━ 29:34 614ms/step - f1_micro: 0.8668 - loss: 0.0406 - precision: 0.8868 - recall: 0.8477